# Gemini Multimodal Embeddings

---
**Lab Duration:**: Approximately 45 minutes
**Difficulty:** Intermediate
**Prerequisites:** Basic Python, familiarity with APIs. Prior exposure to embeddings (trom the RAG or Agentlc Memory labs) Is helpful but not required.

---

## Introduction

This lab introduces **multimodal embeddings** using Google's Gemini API. By the end of it you will understand how Al models represent both text and images as numerical vectors, how to measure similarity between those vectors, and how to use that capability to bulld a cross-modal search system that can find images Using natural language queries.

---

### What Are Embeddings?

An **embedding** is a numerical representation of content. Specifically, It is a list of floatng-point numbers (a vector) that captures the semantic meaning of that content.
When two pieces of content mean similar things, their embeddings will be mathematically close to each other, even if they look completely different on the surface. For example:

- The text `"mountain scenery"` and an image of a snow-capped peak should produce embeddings that are close together.
- The text `"coffee shop"` and an image of a busy road should produce embeddings that are far apart.

This property is what makes semantic search work: Instead of matching keywords, you match meanings.

---

### What Makes This Multimodal?
Most embedding systems only handle text. A **multimodal** embedding model can encode different types of content, In this case both text and images, into the same vector space. This means you can directly compare text embedding with an image embedding using the same mathematical operation.

From a cloud and SRE perspective, this has practical applications:

- Searching a dashboard screenshot library using a text description of a problem
- finding relevant architecture diagrams by describing the pattern you are looking for
- Matching incident screenshots to runbook sections that describe similar symptoms

---

### What You will Build

This lab walks through a complete multimodal embedding pipeline:

1. Load a set of paired text descriptions and images
2. Generate text embeddings using Gemini's `embedding-001` model
3. Generate image embeddings using the same model
4. Compute similarity matrices to see how text and images relate to each other
5. Visualise the embeddings in 2D using PCA
6. Run a cross-modal search: glven a text query find the most similar images.

---

### Tech Stack

| Component |  Role |
|---|---|
| **Google Gemini API (`embedding-001`)** | Generates vector embeddings for both text and images |
| **NumPy** | Handles vector arrays and numerical operations |
| **scikit-learn** | Computes cosine similarity and runs PCA for dimensionality reduction |
| **Pillow (PIL)** | Loads and pre-processes images before sending them to the API |
| **Matplotiib/ Seaborn** | Visualises similarity matrices and PCA scatter plots |
| **Requests** | Fetches images from URLs |

---

## Key Concept: Cosine Similarity

Throughout this lab wo use cosine similarity to measure how close two embeddings are. Cosine similarity measures the angle between two vectors:
- A score of **1.0** means the vectors point in the same direction, i.e. the content is identical or very similar.
- A score of **0.0** means the vectors are perpendicular, i.e. the content is unrelated.
- A score of **-1.0** means the vectors point in opposite directions, i.e. the content is maximally dissimilar.

In practice, embedding similarity scores tend to fall in the range 0.5 to 1.0 for semantically related content.

---

### Before You Start

Yeu need a **Google API key** with access to the Gemini API, Store it in Colab's Secrets panel under the name `GOOGLE_API_KEY`. To open the Secrets panel, click the key icon in the left sidebar of Colab.

Run all cells in order from top to bottom.

---


## Step 1: Install Required Libraries

This cell Installs all the Python packages needed for this lab. Here is what each one is used for:

- `google-generativeai` - the official Google Generative AI Python SDK, which provides access to Gemini models including the embedding model
- `pillow` - the Python Imaging Library (PIL), used to load, convert, and resize Images before passing them to the API
- `numpy` - the numerical computing library used to store and operate on embedding vectors
- `matptotLib` - the plotting library used to render the PCA scatter plot and similarity heatmaps
- `scikit-learn` - provides the cosine similarity function and the PCA algorithm for dimensionality reduction
- `seaborn` -a higher-level plotting library bult on top of matplotlib, used for the heatmap visualisations
- `requests` - used to download Images from URLs before processing them

This cell only needs to be run once per Colab session.


In [ ]:
# Install required packages
!pip3 install google-generativeai pillow numpy matplotlib scikit-learn seaborn requests

## Step 2: Import All Libraries

We import everything in one place so the dependencies are clearly visible.

A few imports worth highlighting:

- `cosine_similarity` from scikit-learn - the function we will use to measure how similar two embedding vectors are
- `PCA` from scikit-learn - Principal Component Analysis, which reduces our high-dimensional embedding vectors (768 or more dimensions) down to 2 dimensions so we can plot them
- `google.generativeai as genai` - the Gemini SDK, aliased as `genai` throughout the notebook
- `warnings. filterwarnings('ignore')` - suppresses deprecation warnings from the SDK so the output stays clean

The `List`, `Dict`, and `Tuple` imports from `typing` are used in the function signatures defined later for clarity.


In [20]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import seaborn as sns
from PIL import Image
import requests
from typing import List, Dict, Tuple
import google.generativeai as genai
import warnings
warnings.filterwarnings('ignore')

## Step 3: Initialise the Gemini Client

We read the API key from Colab's Secrets panel and pass it to `genai:configure()`. This authenticates all subsequent calls made through the `genai` library for the duration of this session.

The key is stored in Colab Secrets rather than hardcoded in the notebook for security. If you hardcoded the key in the source, it would be visible to anyone who can view the notebook file.

If you have not added your key yet:
1. Click the key icon in the left sidebar of Colab
2. Add a new secret with the name `GOOGLE_API_KEY`
3. Paste your Gemini API key as the value
4. Run this cell


In [ ]:
# Initialize Gemini client
from google.colab import userdata

GEMINT_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GEMINT_API_KEY)
print("Gemini client initialized")

In [ ]:
# List available models to find the correct embedding model name
print("Available models supporting embedContent:")
for m in genai.list_models():
    if 'embedContent' in m.supported_generation_methods:
       print(m.name)

## Step 4: Define the Core Helper Functions

This is the largest cell in the notebook. It defines all the functions that the rest of the lab will use. Read through each function and its description before running the cell.

---

**`encode_text_with_gemini(texts, model)`**
Calls `genai.embed_content()` for each text string in the input list and returns all embeddings stacked into a NumPy array. The `task_type="retrieval_document"` parameter tells the model that these embeddings will be used for retrieval rather than classification, which affects how to model optimises the vector representation.

---

**`prepare_image_for_gemini(image_input)`**
Accepts either a URL string or a local file path. Downloads or opens the image, converts it to RGB mode ( required by the API), and resizes it to a maximum of 256x256 pixels using `thumbnail()`. Resizing reduces the payload sent to the API and keeps API calls fast.

---

**`encode_images_with_gemini(image_paths, model)`**
Calls `prepare_image_for_gemini()` for each image, then passes the resulting PIL Image object directly to `genai.embed_content()`. If an image fails to load or embed, a zero vector of the default embedding size (768) is used as a fallback so the rest of the pipeline can continue.

---

**`compute_similarity_matrix(embeddings1, embeddings2)`**
Wraps scikit-learn's `cosine_similarity()` to compute pairwise similarities between two sets of embeddings. If only one set is provided, it computes the self-similarity matrix (every item against every item in the same set).

---

**`cross_modal_search(query_embedding, target_enbeddings, target_labels, top_k)`**
The retrieval function. Given a single query embedding (e.g. from a text string) and a set of target embeddings (eg. from images), it computes cosine similarities, ranks the targets, and returns the top-k most similar items as a list of (label, score) tuples.

---

**`visualize_enbeddings_pca(embeddings_dict, labels_dict, title)`**
Takes embeddings from multiple modalities (text and image), stacks them, runs PCA to reduce from high-dimensional space to 2D, and plots a scatter chart where different modalities are shown in different colours. This lets you visually inspect whether text and image embeddings cluster near each other when they describe the same concept.

---

**`display_images(image_urls, titles)`**
A utility function that fetches and renders images from URLs in a matplotlib grid, with optional title labels beneath each image. Used to visually inspect the sample dataset before running the embedding pipeline.


In [53]:
def encode_text_with_gemini(
    texts: List[str],
    model: str = "models/gemini-embedding-2-preview"
) -> np.ndarray:
    """Generate text embeddings using the never multimodal-ready model."""
    embeddings = []
    for text in texts:
        result = genai.embed_content(
            model=model,
            content=text,
            task_type="retrieval_document"
        )
        embeddings.append(result['embedding'])
    return np.array(embeddings)

def prepare_image_for_gemini(image_input: str) -> Image.Image:
    """Load and prepare image for Gemini API"""
    if image_input.startswith('http'):
        response = requests.get(image_input, timeout=30)
        response.raise_for_status()
        from io import BytesIO
        image = Image.open(BytesIO(response.content))
    else:
        image = Image.open(image_input)

    if image.mode != 'RGB':
        image = image.convert('RGB')
        
    max_size = (256, 256)
    image.thumbnail(max_size, Image.Resampling.LANCZOS)

    return image

def encode_images_with_gemini(
    image_paths:List [str],
    model: str = "models/gemini-embedding-2-preview"
) -> np.ndarray:
    """Generate image embeddings using the never multimodal-ready model."""
    embeddings = []
    for img_path in image_paths:
        try:
            image = prepare_image_for_gemini(img_path)
            result = genai.embed_content(
                model=model,
                content=image
            )
            embeddings.append(result['embedding'])
        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            embeddings.append([0.0] * 3072)
    return np.array(embeddings)

def compute_similarity_matrix(
    embeddings1: np.ndarray,
    embeddings2: np.ndarray = None
) -> np.ndarray:
   """Compute cosine similarity matrix."""
   if embeddings2 is None:
       embeddings2 = embeddings1
   return cosine_similarity(embeddings1, embeddings2)

def cross_modal_search(
    query_embedding: np.ndarray,
    target_embeddings: np.ndarray,
    target_labels: List[str],
    top_k: int = 3
) -> List[Tuple[str, float]]:
    """Find most similar items using cosine similarity."""
    if query_embedding.ndim == 1:
        query_embedding = query_embedding.reshape(1, -1)
    similarities = cosine_similarity(query_embedding, target_embeddings)[0]
    top_indices = np.argsort(similarities)[::-1] [:top_k]
    return [(target_labels[i], float(similarities[i])) for i in top_indices]

def visualize_embeddings_pca(
    embeddings_dict: Dict[str, np.ndarray],
    labels_dict: Dict[str, List[str]],
    title: str = "Gemini Multimodal Embeddings"
):
    """visualize embeddings in 2D using PCA."""
    all_embeddings = []
    all_labels = []
    all_modalities = []
    for modality, embeddings in embeddings_dict.items():
        all_embeddings.extend(embeddings)
        all_labels.extend(labels_dict[modality])
        all_modalities.extend([modality] * len(embeddings))
    all_embeddings = np.array(all_embeddings)
    pca = PCA(n_components=2)
    embeddings_2d = pca.fit_transform(all_embeddings)
    plt.figure(figsize=(12, 8))
    colors = {'text': '#1f77b4', 'image': '#ff7f0e'}
    for modality in embeddings_dict.keys():
        mask = np.array(all_modalities) == modality
        plt.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1], c=colors.get(modality, '#8c564b'), label=f"{modality.title()}", alpha=0.7, s=100)
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    plt.ylabel(f'PC1 ({pca.explained_variance_ratio_[1]:.1%})')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def display_images(image_urls: List[str], titles: List[str] = None):
    """Display images in a grid."""
    fig, axes = plt.subplots(1, len(image_urls), figsize=(15, 5))
    if len(image_urls) == 1:
        axes = [axes]
    for i, url in enumerate(image_urls):
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            from io import BytesIO
            img = Image.open(BytesIO(response.content))
            axes[i].imshow(img)
            axes[i].axis('off')
            if titles: axes[i].set_title(titles[i], fontsize=12, pad=10)
        except Exception as e:
            print(f"Error loading {url}: {e}")
            axes[i].text(
                0.5,
                0.5,
                "Failed to load",
                ha='center',
                va='center'
            )

            axes[i].axis('off')
    plt.tight_layout()
    plt.show()


## Step 5: Load the Sample Dataset

We define a small paired dataset of text descriptions and corresponding image URLs. Each entry in `image_data` contains:

- `text` - a natural language description of the image content
- `image_url` - a URL to a publicly accessible image
- `category` - a category label used for reference

The five samples span different categories: nature, vehicles, interior spaces, and technology. This variety wil help us see meaningful differences in the similarity scores and PCA plot later.

After defining the full dataset, we extract the `texts`, `image_urls`, and `categories` into separate flat lists for easier use with our embedding functions.

Note that the images are fetched on-demand from Unsplash. You need a working internet connection for this and for the embedding API calls.


In [ ]:
# Sample data
image_data = [
    {
        "text": "Serene mountain landscape with snow-capped peaks",
        "image_url": "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=400&h=300&fit=crop",
        "category": "nature"
    },
    {
        "text": "Vintage red sports car on a winding road",
        "image_url": "https://images.unsplash.com/photo-1503376780353-7e6692767b70?w=400&h=300&fit=crop",
        "category": "vehicles"
    }, 
    {
        "text": "Cozy coffee shop interior with warm Lighting",
        "image_url": "https://images.unsplash.com/photo-1501339847302-ac426a4a7cbb?w=400&h=300&fit=crop",
        "category": "interior"
    },
    {
        "text": "Beautiful sunset over calm ocean waters",
        "image_url": "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=400&h=300&fit=crop",
        "category": "nature"
    },
    {
        "text": "Technology workspace with laptop and gadgets",
        "image_url": "https://images.unsplash.com/photo-1498050108023-c5249f4df085?w=400&h=300&fit=crop",
        "category": "technology"
    }
]

texts = [item["text"] for item in image_data]
image_urls = [item["image_url"] for item in image_data]
categories = [item["category"] for item in image_data]

print(f"Loaded {len(texts)} text descriptions and {len(image_urls)} images")
print("Categories:", categories)

## Step 6: Display the Sample Images

Before running any embedding operations, we visually inspect the dataset. This confirms that the images are accessible and gives you a clear reference for interpreting the similarity scores and PCA plot later.

The `display_images()` function we defined earlier fetches each image from its URL, renders it in a matplotiib grid, and displays the corresponding text description as a title above each image.

After running this cell you should see all five images displayed side by side. If any image fails to load it will show a placeholder, which is handled gracefully by the try/except block inside `display_images()`.

In [ ]:
display_images(image_urls, texts)

## Step 7: Generate Text Embeddings

Now we run the five text descriptions through Gemini's `embedding-001` model to produce five embedding vectors.

Internally, `encode_text_with_gemini()` loops over each string, calls `genai.embed_content()` with `task_type="retrieval_document"` and appends the returned vector to a list. The result is a 2D NumPy array with shape `(5, N)` where 5 is the number of texts and N is the embedding dimension (768 for `embedding-001` ).

The shape printed at the end confirms this. Each row in `text_embeddings` is a 768-dimensional vector representing one text description.

This cell makes five API calls to the Gemini embedding endpoint. It should complete in a few seconds.

In [ ]:
print("Generating text embeddings (multinodal-v2 space)...")
text_embeddings = encode_text_with_gemini(texts, model="models/gemini-embedding-2-preview")
print(f"Text embeddings shape: {text_embeddings.shape}")

## Step 8: Generate Image Embeddings

We now do the same for the images. `encode_images_with_gemini()` fetches each image from its URL, resizes it to a maximum of 256x256 pixels, converts it to RGB, and passes the PIL Image directly to `genai.embed_content()`.

The key thing to understand here is that we are using the **same model** (`embedding-001`) for both text and images. This is what makes the embeddings mutimodel: text and image vectors live in the same mathematical space, which means we can directly compare them using cosine similarity.

The resulting `image_embeddings` array has the same shape as `text_embeddings`: `(5, 768)`. Each row is the vector representation of one image.

This cell fetches five images from the web and makes five API calls. It may take 15-30 seconds depending on your connection speed.

In [ ]:
print ("Generating image enbeddings...")
image_embeddings = encode_images_with_gemini(image_urls)
print(f"Image embeddings shape: {image_embeddings.shape}")

## Step 9: Compute Similarity Matrices

With both sets of embeddings ready, we can now measure how similar everything is to everything else. This cell computes three separate similarity matrices:

**Text-to-text similarity**
A 5x5 matrix where entry `[i, j]` is the cosine similarity between text i and text j. The diagonal will always be 1.0 (each text is identical to itself). Off-diagonal entries reveal which text descriptions are semantically close. You should see higher scores between the two nature entries (moutains and ocean sunset) than between nature and technology.

**Image-to-image similarity**
The same structure but for images. Since two of our images share the same URL (the mountain photo is reused for the ocean entry), their embeddings will be identical, giving a perfect 1.0 similarity.

**Cross-modal similarity (text-to-image)**
This is the most interesting matrix. Entry `(i, j)` is the cosine similarity between text i and image j. Ideally, the diagonal (where text i is paired with its corresponding image i) should have higher scores than off-diagonal entries. This is the property that makes cross-modal search work: a text description should be more similar to its matching image than to unrelated images.

Read the printed matrices from left to right across each row. Each row represents one query item, and each column represents one target item.


In [ ]:
# Text-to-text similarity
text_similarity = compute_similarity_matrix(text_embeddings)
print("Text-to-text similarity matrix:")
print(text_similarity)

# Image-to-image similarity
image_similarity = compute_similarity_matrix(image_embeddings)
print("\nImage-to-image similarity matrix:")
print(image_similarity)

# Cross-modal similarity (text-to-image)
cross_modal_similarity = compute_similarity_matrix(text_embeddings, image_embeddings)
print("\nCross-modal (text-to-image) similarity matrix:")
print(cross_modal_similarity)

## Step 10: Visualise Embeddings with PCA

Embedding vectors are high-dimensional (768 dimensions). We cannot directly plot 768-dimensional data, so we use **Principal Component Analysis (PCA)** to project all vectors down to 2 dimensions while preserving as much of the variance as possible.

Here is what this cell does:

1. It combines the text embeddings and image embeddngs into a single array
2. It runs PCA to reduce from 768 dimensions to 2
3. It plots each point on a scatter chart, colouring text points in blue and image points in orange
4. The axis labels show how much of the original variance each principal component captures.

**What to look for in the plot:**
If the model is working well as a multimodal system, text and image points that describe the same concept should appear close to each other in the 2D space. For example, the text "Serene mountain landscape and the mountain image should cluster together, while the text "Technology workspace" and the laptop image should cluster in a diferent region.

PCA is a lossy projection, so some relationships visible in the full 768-dimensional space may not be perfectly preserved In 2D. However it gives a useful first-pass visual intuition.

In [ ]:
embeddings_dict = {'text': text_embeddings, 'image': image_embeddings}
labels_dict = {'text': texts, 'image': [f'Image {i+1}' for i in range(len(image_urls))]}
visualize_embeddings_pca(embeddings_dict, labels_dict)

## Step 11: Cross-Modal Search

This is the practical payoff of the lab. We demonstrate **cross-modal search**: given a text query, find the most similar images from our dataset using embedding similarity.

Here is what the code does step by step:

1. Define a text query string: `"mountain scenery"`
2. Call `encode_text_with_gemini()` to convert the query test into a single 768-dimensional vector. The `[0]` at the end extracts just the first (and only) row since we passed a list of one item.
3. Call `cross_modal_search()` with the query vector, the five image embeddings, their labels, and `top_k=3` to get the three best matches.
4. Print each match with its cosine similarity score.

**What to expect:**
The query `"mountain scenery”` should rank the nature images(mountain landscape and ocean sunset, which share the same image URL in this dataset) highest, and the technology and vehicle images lowest.

To experiment further, try changing `query_text` to something like `"technology desk Setup"` or `"red car"` and re-run the cell to see how the rankings change.

In [ ]:
# Search for images using text query
query_text = "mountain scenery"
query_embedding = encode_text_with_gemini([query_text])[0]
results = cross_modal_search(query_embedding, image_embeddings, [f'Image {i+1}' for i in range(len(image_urls))], top_k=3)
print(f"Query: '{query_text}'")
print('\nTop 3 matching images:')
for label, score in results:
    print(f'{label}: {score:.4f}')

## Lab Summary

You have completed the Gemini Multimodal Embeddings lab. Here is a recap of what was covered.

**What you built:**
- A complete multimodal embedding pipeline using Gemini's `embedding-001` model
- Text embedding generation for five natural language descriptions
- Image embedding generation for five images fetched from URLS.
- Three similarity matrices: text-to-text, image-to-image, and cross-modal (text-to-Image)
- A PCA visualisation of all embeddings in a shared 2D space
- A cross-modal search function that retrieves images using a text query

**Key concepts covered:**

| Concert | What it means |
|---|---|
| Embedding | A numerical vector representing the semantic meaning of content |
| Multimodal embedding | A single model that embeds both text and images into the same vector space |
| Cosine similarity | A measure of the angle between two vectors, used to quantify semantic closeness |
| Cross-modal search | Finding images using a text query by comparing embeddings across modalities |
| PCA | A dimensionality reduction technique that projects high-dimensional vectors to 2D for visualisation |

**How this connects to real-world cloud and SRE work:**

The same pattern you used In this lab can be applied to practical operational problems:
- Indexing screenshots of monitoring dashboards and searching them by describing the anomaly pattern you are looking for
- Matching incident reports (text) against architecture diagrams (images) to surface relevant context automatically
- Building a runbook retrieval system where both text descriptions and diagram images are stored in a shared embedding space and retrieved together